# Model Development

## 1. Loading and Splitting datasets


We are going to reload the cleaned and realy datasets from previous step, and split them into 80-20 ratio of train-test. Since the datasets are bias (only 6% Fraud ratio), we are using stratify to make sure the are split evenly with the same fraud ratio.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df_full = pd.read_csv('data/processed/claims_full.csv')
df_selected = pd.read_csv('data/processed/claims_selected.csv')


In [2]:
df_selected.columns == 'FraudFound'

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False,  True,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False])

In [3]:

# separate features and target for each dataset
X_full = df_full.drop(columns='FraudFound')
y_full = df_full['FraudFound']

X_selected = df_selected.drop(columns='FraudFound')
y_selected = df_selected['FraudFound']

# Split - stratify ensures same froud ratio in train and test
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

X_selected_train, X_selected_test, y_selected_train, y_selected_test = train_test_split(
    X_selected, y_selected, test_size=0.2, random_state=42, stratify=y_selected)

## 2. Train Baseline Model

Our baseline model chosen for this is Logistic Regression and the reasons are as below:
- Simple,fast and interpretable - good starting point nefore we try complex models
- Works well for binary classification (`Fraud` vs `Not Fraud`)
- Coefficients are interpretable - we can see which features push toward fraud

However there are some limitations using this model:
- Assumes linear relatinship between features and the log-odds of fraud
- Sensitive to feature scaling - we will show how using `StandardScalar` before fitting might impact our model performance

We are going to move stage by stage in evaluating and comparing the performance  of our logistic regression model trained on both datasets. We will focus on **F1-score** and **ROC-AUC** as our primary metrics.

F1-score balances precision and recall which is important in fraud detection where false negatives are costly. A false negative occurs when the model predict a claim as legitimate when it is actually fraudulent -- meaning the insurance company processes and pays out a fraudulent claim which leads to financial loss to the company. Therefore, minimising false negatives is critical, which is why **recall**(recall measures how well the model cathces actual fraud cases) is an important consideration.

ROC-AUC on the other hand measures how well the model distinguishes between fraudulent and legitimate claims overall.

### 2.1 Stage 1 - Base Model

We are going to use Linear Regression with no  **class-weight** fitting and feature scaling.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

#train on full features
lr_full_s1 = LogisticRegression(max_iter=1000, random_state = 42)
lr_full_s1.fit(X_full_train, y_full_train)

#train on selected features
lr_selected_s1 = LogisticRegression(max_iter=1000, random_state = 42)
lr_selected_s1.fit(X_selected_train, y_selected_train)

# Predictions on test sets
y_full_pred_s1 = lr_full_s1.predict(X_full_test)
y_selected_pred_s1 = lr_selected_s1.predict(X_selected_test)

# Predicted probabilities (class 1 = fraud)
y_full_proba_s1 = lr_full_s1.predict_proba(X_full_test)[:,1]
y_selected_proba_s1 = lr_selected_s1.predict_proba(X_selected_test)[:,1]

# Evaluate
print("=== Full Features ===")
print(classification_report(y_full_test, y_full_pred_s1))
print("ROC-AUC:", round(roc_auc_score(y_full_test, y_full_proba_s1),2))
print("Confusion Matrix:")
print(confusion_matrix(y_full_test,y_full_pred_s1))

print("=== Selected Features ===")
print(classification_report(y_selected_test, y_selected_pred_s1))
print("ROC-AUC:", round(roc_auc_score(y_selected_test, y_selected_proba_s1),2))
print("Confusion Matrix:")
print(confusion_matrix(y_full_test,y_selected_pred_s1))


/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Full Features ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      2899
           1       0.00      0.00      0.00       185

    accuracy                           0.94      3084
   macro avg       0.47      0.50      0.48      3084
weighted avg       0.88      0.94      0.91      3084

ROC-AUC: 0.73
Confusion Matrix:
[[2899    0]
 [ 185    0]]
=== Selected Features ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      2899
           1       0.00      0.00      0.00       185

    accuracy                           0.94      3084
   macro avg       0.47      0.50      0.48      3084
weighted avg       0.88      0.94      0.91      3084

ROC-AUC: 0.74
Confusion Matrix:
[[2899    0]
 [ 185    0]]


/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/hom

There are all 0 values for precision, recall and f1-score for class 1 meaning that this model cannot find any fraudulent claims. 

### 2.2 Stage 2 - Addressing Class Imbalance

We are going to use Linear Regression with **class_weight**=balance fitting.

In [5]:
#train on full features
lr_full_s2 = LogisticRegression(max_iter=1000, random_state = 42, class_weight='balanced')
lr_full_s2.fit(X_full_train, y_full_train)

#train on selected features
lr_selected_s2 = LogisticRegression(max_iter=1000, random_state = 42, class_weight='balanced')
lr_selected_s2.fit(X_selected_train, y_selected_train)

# Predictions on test sets
y_full_pred_s2 = lr_full_s2.predict(X_full_test)
y_selected_pred_s2 = lr_selected_s2.predict(X_selected_test)

# Predicted probabilities (class 1 = fraud)
y_full_proba_s2 = lr_full_s2.predict_proba(X_full_test)[:,1]
y_selected_proba_s2 = lr_selected_s2.predict_proba(X_selected_test)[:,1]

# Evaluate
print("=== Full Features ===")
print(classification_report(y_full_test, y_full_pred_s2))
print("ROC-AUC:", round(roc_auc_score(y_full_test, y_full_proba_s2),2))
print("Confusion Matrix:")
print(confusion_matrix(y_full_test,y_full_pred_s2))

print("=== Selected Features ===")
print(classification_report(y_selected_test, y_selected_pred_s2))
print("ROC-AUC:", round(roc_auc_score(y_selected_test, y_selected_proba_s2),2))
print("Confusion Matrix:")
print(confusion_matrix(y_full_test,y_selected_pred_s2))


/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Full Features ===
              precision    recall  f1-score   support

           0       0.97      0.64      0.77      2899
           1       0.11      0.68      0.18       185

    accuracy                           0.64      3084
   macro avg       0.54      0.66      0.48      3084
weighted avg       0.92      0.64      0.73      3084

ROC-AUC: 0.72
Confusion Matrix:
[[1845 1054]
 [  59  126]]
=== Selected Features ===
              precision    recall  f1-score   support

           0       0.97      0.64      0.78      2899
           1       0.12      0.72      0.20       185

    accuracy                           0.65      3084
   macro avg       0.54      0.68      0.49      3084
weighted avg       0.92      0.65      0.74      3084

ROC-AUC: 0.74
Confusion Matrix:
[[1869 1030]
 [  51  134]]


/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Applying `class_weight='balanced'` telss the model to penalise misclassification of fraud cases more heavily,allowing it to start detecting the minority class instead of defaulting everything as legitimate.

The result show that the model now cathes fraud cases - recall improve from 0.00 to 0.72 for slected features.. However precision remain low at 0.12, meaning the model flags many legitimate claims as fraudulent (1030 false positives). Selected features outperform full features across all metrics,confirming that removing non-significant features reduces noise. 

### 2.3 Stage 3 - Feature Scaling

Feature scaling will be applied in this stage to further improve performance.

In [6]:
from sklearn.preprocessing import StandardScaler

scaler_full = StandardScaler()
scaler_selected = StandardScaler()

X_full_train_std = scaler_full.fit_transform(X_full_train)
X_full_test_std = scaler_full.transform(X_full_test)

X_selected_train_std = scaler_selected.fit_transform(X_selected_train)
X_selected_test_std = scaler_selected.transform(X_selected_test)


In [9]:
#train on full features
lr_full_s3 = LogisticRegression(max_iter=1000, random_state = 42, class_weight='balanced')
lr_full_s3.fit(X_full_train_std, y_full_train)

#train on selected features
lr_selected_s3 = LogisticRegression(max_iter=1000, random_state = 42, class_weight='balanced')
lr_selected_s3.fit(X_selected_train_std, y_selected_train)

# Predictions on test sets
y_full_pred_s3 = lr_full_s3.predict(X_full_test_std)
y_selected_pred_s3 = lr_selected_s3.predict(X_selected_test_std)

# Predicted probabilities (class 1 = fraud)
y_full_proba_s3 = lr_full_s2.predict_proba(X_full_test_std)[:,1]
y_selected_proba_s3 = lr_selected_s2.predict_proba(X_selected_test_std)[:,1]

# Evaluate
print("=== Full Features ===")
print(classification_report(y_full_test, y_full_pred_s3))
print("ROC-AUC:", round(roc_auc_score(y_full_test, y_full_proba_s3),2))
print("Confusion Matrix:")
print(confusion_matrix(y_full_test,y_full_pred_s3))

print("=== Selected Features ===")
print(classification_report(y_selected_test, y_selected_pred_s3))
print("ROC-AUC:", round(roc_auc_score(y_selected_test, y_selected_proba_s3),2))
print("Confusion Matrix:")
print(confusion_matrix(y_full_test,y_selected_pred_s3))




=== Full Features ===
              precision    recall  f1-score   support

           0       0.99      0.61      0.75      2899
           1       0.13      0.88      0.22       185

    accuracy                           0.62      3084
   macro avg       0.56      0.74      0.49      3084
weighted avg       0.94      0.62      0.72      3084

ROC-AUC: 0.76
Confusion Matrix:
[[1759 1140]
 [  22  163]]
=== Selected Features ===
              precision    recall  f1-score   support

           0       0.99      0.60      0.75      2899
           1       0.13      0.90      0.22       185

    accuracy                           0.62      3084
   macro avg       0.56      0.75      0.49      3084
weighted avg       0.94      0.62      0.72      3084

ROC-AUC: 0.76
Confusion Matrix:
[[1752 1147]
 [  19  166]]


/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


Applying `StandardScaler`  standardises all numerical features to the same scale, which is important for logistic regression as it is sensitive to feature magnitude. the scaler is fitted on the training set only and applied to both train and test sets to prevent data leakage.

Scaling further improve recall significantly - from 0.72 to 0.90 for selected features, meaning the model now correctly identify 90% of actual fraud cases. Both datasets perform similarly after scaling, though selected features still show slightly better in recall.

Precision remains low at 0.13,which is expected - this is the ***precision-recall tradeoff*. By prioritising recall (catching fraud), the model casts a wider net and flags more legitimate claims as fraudulent. We can tune this tradeoff via the classification threshold in later iterations. For fraud detection, high recall is the priority as missiing a fraud case leads to direct financial loss for the company.

## 3. Model Evaluation Summary

The table below combines the metrics from all three stages into one view for easier comparison. From the previous steps, we found that the model with feature scaling and `class_weight='balanced'` performs best. Going forward, we'll focus on the selected feature dataset, as it consistently outperforms the full feature set - likely due to reduced noise from irrelevant features.



In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score

def get_metrics(y_true,y_pred, y_proba):
    return {"Recall": recall_score(y_true,y_pred),
            "Precision": precision_score(y_true,y_pred),
            "F1-score": f1_score(y_true,y_pred),
            "ROC-AUC": roc_auc_score(y_true,y_proba)}
results = []

# Stage 1
results.append({"Stage":"Stage 1 - Base LR",
                "Feature Set":"Full", 
                **get_metrics(y_full_test, y_full_pred_s1, y_full_proba_s1)}) # ** is dict unpacking operator
results.append({"Stage":"Stage 1 - Base LR",
                "Feature Set":"Selected", 
                **get_metrics(y_selected_test, y_selected_pred_s1, y_selected_proba_s1)}) 

# Stage 2
results.append({"Stage":"Stage 2 - class_weight='balanced'",
                "Feature Set":"Full", 
                **get_metrics(y_full_test, y_full_pred_s2, y_full_proba_s2)}) 
results.append({"Stage":"Stage 2 - class_weight='balanced'",
                "Feature Set":"Selected", 
                **get_metrics(y_selected_test, y_selected_pred_s2, y_selected_proba_s2)}) 

# Stage 3
results.append({"Stage":"Stage 3 - + StandardScalar",
                "Feature Set":"Full", 
                **get_metrics(y_full_test, y_full_pred_s3, y_full_proba_s3)}) 
results.append({"Stage":"Stage 3 - + StandardScalar",
                "Feature Set":"Selected", 
                **get_metrics(y_selected_test, y_selected_pred_s3, y_selected_proba_s3)}) 

summary_df = pd.DataFrame(results)
summary_df = summary_df.round(2)
summary_df

/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/carclaims/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Stage,Feature Set,Recall,Precision,F1-score,ROC-AUC
0,Stage 1 - Base LR,Full,0.00,0.00,0.00,0.73
1,Stage 1 - Base LR,Selected,0.00,0.00,0.00,0.74
2,Stage 2 - class_weight='balanced',Full,0.68,0.11,0.18,0.76
3,Stage 2 - class_weight='balanced',Selected,0.72,0.12,0.20,0.76
4,Stage 3 - + StandardScalar,Full,0.88,0.13,0.22,0.76
5,Stage 3 - + StandardScalar,Selected,0.90,0.13,0.22,0.76
